> 在线讲义：[第 11 章 · SuperMimic 全流程：三步循环、关键超参、结果和怎么选路线](https://wikieden.github.io/robot-motion-rl/lessons/11-supermimic.html) · 本书《机器人运控与强化学习》

# 第 11 章 · SuperMimic 全流程：三步循环、关键超参、结果和怎么选路线

配套讲义：仓库里 `robot-motion-rl/notes/lessons/11-supermimic.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（超参全部取自 SuperMimic 表 A.5）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 缓冲区 | 4096 × 1000 | `buffer_size()` | 4,096,000 |
| 第二步 WM 更新 | 10 × 2048 × 8 | `wm_transitions_per_update()` | 163,840 |
| 第二步 策略更新 | 4096 × 32 | `imagined_steps_per_update()` | 131,072；× 32 万 ≈ 4.19e10 |

超参来自 SuperMimic 附录表 A.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
HP = {
    "envs_per_gpu": 4096,
    "buffer_steps_per_env": 1000,
    "window_k": 8,
    "wm_batches": 10,
    "wm_windows_per_batch": 2048,
    "horizon_H": 32,
    "imagined_rollouts": 4096,
    "iterations": 320_000,
}

**函数 `buffer_size`**：回放缓冲区能存多少条转移：环境数 × 每环境步数。

In [ ]:
def buffer_size(hp=HP):
    """回放缓冲区能存多少条转移：环境数 × 每环境步数。"""
    return hp["envs_per_gpu"] * hp["buffer_steps_per_env"]

**函数 `wm_transitions_per_update`**：一次 world model 更新看多少个转移：10 批 × 2048 个窗口 × 8 步。

In [ ]:
def wm_transitions_per_update(hp=HP):
    """一次 world model 更新看多少个转移：10 批 × 2048 个窗口 × 8 步。"""
    return hp["wm_batches"] * hp["wm_windows_per_batch"] * hp["window_k"]

**函数 `imagined_steps_per_update`**：一次策略更新想象多少步：4096 条 × 32 步。

In [ ]:
def imagined_steps_per_update(hp=HP):
    """一次策略更新想象多少步：4096 条 × 32 步。"""
    return hp["imagined_rollouts"] * hp["horizon_H"]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 回放缓冲区")
print(f"4096 × 1000 = {buffer_size():,} 条转移")

In [ ]:
print("\n第二步 每次迭代的计算量")
print(f"world model：10 × 2048 × 8 = {wm_transitions_per_update():,} 步预测")
print(f"策略：4096 × 32 = {imagined_steps_per_update():,} 步想象")
print(f"32 万次迭代共想象 {imagined_steps_per_update() * HP['iterations']:.2e} 步")

In [ ]:
print("\n第三步 三个提速因素")
print("想象比仿真快 >10×（64k 并行时 62×）；解析梯度比 PPO-WM 省约 64× 想象步数；误差随数据增长保持 0.007–0.011")

## 3. 代码里才看得到的

- 这一章的「代码对照」只是算账：完整的三步循环要 GPU 仿真器和几个大网络，见学习计划 MC-M16（HalfCheetah 上的小规模版本，本机可做）和 MC-M17（`[GPU]`）。

## 4. 自己改一改

- 把 `horizon_H` 改成 128（论文里一张 H100 装得下的最长视野），看每次想象的步数涨几倍。
- 估一下：如果每步想象花 1 微秒，32 万次迭代的想象部分要多少小时？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 上路采数据、更新车感、躺床上演练，三步每轮都转；
> 旧数据照样能用，物理不看是谁采的；
> 缓冲四百万、每轮想十三万，一共四百多亿步；
> 共享模型、想象快、梯度准，三样乘起来就是几百倍。